In [ ]:
import pandas as pd
import Bio as bp
import requests
from bs4 import BeautifulSoup
import re
import time
from collections import defaultdict
import numpy as np
from Bio import SeqIO

In [ ]:
filename = 'Hari.csv'       #Proteomics peptides file
prenyldata_file = 'cardiomyocytes_2.xlsx'       #File with statistically significant positively enriched proteins. We will henceforth call this file the volcano plot file.
prenylated_file = 'Total prenylation list - Human.csv'      #File with bonafide prenylated proteins

In [ ]:
input = pd.read_csv(filepath_or_buffer=filename)          #Read files
prenyldata= pd.read_excel(prenyldata_file)
prenylated= pd.read_csv(prenylated_file)

In [ ]:
input_cleaned = input.dropna(subset=['Proteins'])           #Drop rows with no UniProt ID
input_cleaned = input_cleaned.dropna(subset=['End position'])       #Drop rows with NaN values under 'End position' column

In [ ]:
#Some peptides might be associated with more than one UniProt ID separated by semi-colons. This block of code will chose the first UniProtID. This is an arbitrary choice.

for i in input_cleaned.index:
    if ";" in input_cleaned.loc[i, 'Proteins']:
        split = input_cleaned.loc[i, 'Proteins'].split(";")
        input_cleaned.loc[i, 'Proteins'] = split[0]
    else:
        pass 

In [ ]:
prenyldata = prenyldata.dropna(subset=['Significant'])      #Drop rows with NaN values under 'Siginificant' column

In [ ]:
#This block of code will drop all proteins that are not significantly enriched

for i in prenyldata.index:
    if "+" in prenyldata.loc[i, 'Significant']:
        pass
    else:
        prenyldata = prenyldata.drop(index=i)

#Some proteins might be associated with more than one UniProt ID separated by semi-colons. This block of code will chose the first UniProtID. This is an arbitrary choice.

for i in prenyldata.index:
    if ";" in prenyldata.loc[i, 'Protein IDs']:
        split = prenyldata.loc[i, 'Protein IDs'].split(";")
        prenyldata.loc[i, 'Protein IDs'] = split[0]
    else:
        pass 

known = list(prenylated['Uniprot ID'])          #Creates a list of bonafide prenylated proteins

#This block of code will drop all bonafide prenylated proteins from the volcano plot file.

for i in prenyldata.index:
    if prenyldata.loc[i, 'Protein IDs'] in known:
        prenyldata = prenyldata.drop(index = i)
    else:
        pass

In [ ]:
significantproteins = list(prenyldata['Protein IDs'])
input_cleaned = input[input['Proteins'].isin(significantproteins)].copy()       #Filters out non-bonafide prenylated significantly enriched proteins

In [ ]:
#This block of code will fetch UniProtIDs for all non-bonafide prenylated significantly enriched proteins

input_genes = list(input_cleaned['Proteins'])

print(f"Total unique IDs to fetch: {len(input_genes)}")


submit_url = "https://rest.uniprot.org/idmapping/run"
payload = {
    'from': 'UniProtKB_AC-ID',
    'to': 'UniProtKB',        # Keep this target database
    'ids': ','.join(input_genes) 
}

print("Submitting job to UniProt...")
response = requests.post(submit_url, data=payload)
response.raise_for_status()

job_id = response.json()['jobId']
print(f"Job successfully created! Job ID: {job_id}")


status_url = f"https://rest.uniprot.org/idmapping/status/{job_id}"
while True:
    status_response = requests.get(status_url)
    status_response.raise_for_status()
    status_data = status_response.json()
    
    if "results" in status_data or status_response.status_code == 200:
        print("Job complete!")
        break
    
    print("Job is processing... waiting 5 seconds.")
    time.sleep(5)


results_url = f"https://rest.uniprot.org/idmapping/uniprotkb/results/stream/{job_id}?format=fasta"
print("Downloading FASTA data...")
results_response = requests.get(results_url)

if results_response.status_code != 200:
    print(f"Stream failed: {results_response.text}")
    results_response.raise_for_status()

results = results_response.text

#Save data
with open(f"{filename}_{prenyldata_file}_uniprot_results.fasta", "w") as f:
    f.write(results)

print(f"\nFinished! Saved to '{filename}_{prenyldata_file}_uniprot_results.fasta'. Preview:")
print(results[:500])

In [ ]:
records = list(SeqIO.parse(f"{filename}_{prenyldata_file}_uniprot_results.fasta", "fasta"))           #Parse through above FASTA file and stores each protein as a record. All records are stored in a list.
recordsx = SeqIO.to_dict(SeqIO.parse(f"{filename}_{prenyldata_file}_uniprot_results.fasta", "fasta"))

In [ ]:
end_positions = dict()      #dictionary to store the last tryptic cleavage site
input_genes_set = set(input_genes)          #set to make sure we check for each protein only once

#The following FOR blocks will add the end positions of all tryptic cleavage sites for a given protein and store the as key-value pairs in a dictionary

for protein in input_genes_set:
    for protein1 in input_genes_set:
        
        if protein1 == protein:
            
            
            
            numbers = []
            index1 = input_cleaned.index[input_cleaned["Proteins"] == protein1]
            for index2 in index1:
                numbers.append(input_cleaned.at[index2, 'End position'])
                end_positions.update({protein1 : numbers})
            
        else:
            continue  

In [ ]:
#The following FOR block will pick out the last tryptic cleavage site for a protein and update the above dictionary. 
#Each key (UniProtID) should now only have one value (last tryptic cleavage site) in the dictionary.

for key, value in end_positions.items():
    value = [max(value)]
    end_positions[key] = value

In [ ]:
records = list(SeqIO.parse(f"{filename}_{prenyldata_file}_uniprot_results.fasta", "fasta"))
Cys_list = []           #List to store all proteins that have a CaaX box after the last tryptic cleavage site
Cys_dict = defaultdict(list)       #Dictionary to store all proteins that have a CaaX box after the last tryptic cleavage site along with the sequence after the cleavage site
Caa = []
#aa = ['A', 'G', 'V', 'L' 'M' 'I']      #list of aliphatic amino acids
aa = ["A","R","N","D","E","Q","G","H","I","L","K","M","F","P","S","T","W","Y","V"]      #All 20 canonical amino acids exccept for cysteine  

#This following block of code makes a list of all combination of the tripeptide 'Caa'
for x in aa:
    for y in aa:
        Caa.append(f"C{x}{y}")

#For each protein, the following FOR block will check if any combination of Caa is present in the protein after the last tryptic cleavage site. 
# If present, the protein along with the sequence after the last cleavage site is store as a key-value pair in a dictionary and UniProtIDs in a list.
#If there is no Caa sequence present, the protein will be ignored

for key, value in end_positions.items():
    
    if np.isnan(value[0]) == True:
       end_positions[key] = [0]
    else:
        value[0] = int(value[0])

for key, value in end_positions.items():
    for record in records:
        if key == record.id.split('|')[1]:
            for boop in Caa:
                if boop in str(record.seq[value[0]:]):
                    Cys_list.append(key)
                    Cys_dict[key].append(record.seq[value[0]:])
                else:
                    pass
        else:
            pass


In [ ]:
# Use defaultdict so we can append multiple sequences to the same ID
cys_sequences_dict = defaultdict(list)

cys_set = set(Cys_list)

for id1, seq1 in recordsx.items():
    prot_id = id1.split('|')[1]
    
    if prot_id in cys_set:
        seq = str(seq1.seq)
        cleavage_pos = int(end_positions[prot_id][0])
        tail = seq[cleavage_pos:]
        
        for boop1 in Caa:
            if boop1 in tail:
                # Calculate position for this specific motif
                Caa_pos = cleavage_pos + tail.find(boop1)
                
                # Highlight just this specific CaaX box
                seq_colored = f"{seq[:Caa_pos].lower()}{seq[Caa_pos:Caa_pos+4].upper()}{seq[Caa_pos+4:].lower()}"
                
                # Append to the list for this protein
                cys_sequences_dict[prot_id].append(seq_colored)

# To view the results:
# for key, seq_list in cys_sequences_dict.items():
#     for seq_colored in seq_list:
#         print(f"{key}: {seq_colored}")

In [ ]:
Cys_dict_trimmed = defaultdict(set) #Dictionary to store key(UniProtID) - value(Trimmed sequence). The trimmed sequence cuts off all amino acids after the CaaX box.
justthecaax = defaultdict(set)
#The following FOR block trims the peptide sequence after the CaaX box. 

for key1, value1 in Cys_dict.items():
    for boop1 in Caa:
        for i in range(len(value1)):
            if boop1 in str(value1[i]):
                Caa_pos = str(value1[i]).find(boop1)
                Caax = str(value1[i][Caa_pos:Caa_pos+4])
                if len(Caax) < 4:
                    continue
                else:
                    Cys_dict_trimmed[key1].add(str(value1[i][:Caa_pos+4]))
                    justthecaax[key1].add(str(value1[i][Caa_pos:Caa_pos+4]))
                        
            else:
                pass

#The Prenylation Prediciton Suite does not accept peptide sequences that are shorter than 15 amino acids long. 
# This following block of code will append 15 lysine residues to any peptides that is shorter than 15 amino acids long.

for key, value in Cys_dict_trimmed.items():
    value = list(value)
    for seq in value:
        if len(str(seq)) < 15:
            seq1 = ('K' * 15) + seq
            seq_index = value.index(seq)
            value[seq_index] = seq1
            Cys_dict_trimmed.update({key:value})
        else:
            pass

In [ ]:
#Use this block of code to see which protein has a given CaaX box
#key = list(justthecaax.keys())[list(justthecaax.values()).index({"CLV"})]
#print(key)  

In [ ]:
api_url = "https://mendel.imp.ac.at/PrePS/cgi-bin/clusterBlastPrePS2.cgi"           #Link to submit sequence to Prenylation Predicition Suite (PrePS)
Final_list = defaultdict(list)

#The following FOR block will submit the trimmed sequence for all peptides with a CaaX box into PrePS in FASTA format

for key2, value2 in Cys_dict_trimmed.items():
    value2_list = list(value2)
    for i in range(len(value2_list)):
        seq = f">{key2}\n{value2_list[i]}"
        fasta_data = seq
                




        payload = {
            "Sequence": fasta_data,
            "FT": "ON",
            "GGT1": "ON",
            "GGT2": "ON"
        }


        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Referer": "https://mendel.imp.ac.at/PrePS/"
        }

        response = requests.post(api_url, data=payload, headers=headers)

    #The HTML output is then parsed to get prenylation scores for FTase and GTase I
    #The UniProt ID for a protein along with it's FTase and GTase I scores in stored in Final_list as two separate lists respectively.

        soup = BeautifulSoup(response.text, 'html.parser')
        soup_string = str(soup.get_text(separator='\n', strip=True))
        text = soup_string
        for m in re.finditer(r'Score:\s*([0-9.-]+)', text):
            
            Final_list[key2].append(f"{text[m.start()+7:m.start()+12]}")


In [ ]:
FTase = defaultdict(list)          #List to store all FTase scores
GGTase = defaultdict(list)         #List to store all GTase I scores

#The following FOR block will assort FTase and GGTase I prenylation scores into two separate lists

for key, value in Final_list.items():
    for score in value:
        if value.index(score) % 2 == 0:
            FTase[key].append(score)
        else:
            GGTase[key].append(score)
            
#for values in FTase.values():
    #values = values.reverse()
#for values in GGTase.values():
    #values = values.reverse()
    

In [ ]:
IDs = []        #List to store UniProtIDs

#The following FOR block will grab UniProtIDs for each of the proteins in Final_list

for key, value in Final_list.items():
    IDs.append(key)
    IDs = IDs

In [ ]:
final_FTase_csv = pd.DataFrame.from_dict(pd.Series(FTase))
final_GGTase_csv = pd.DataFrame.from_dict(pd.Series(GGTase))
final_seq_csv = pd.DataFrame.from_dict(pd.Series(justthecaax))
final_csv = pd.DataFrame()
final_csv['FTase scores'] = final_FTase_csv.iloc[:, 0]
final_FTase_csv['GGTase scores'] = final_GGTase_csv.iloc[:, 0]

       #Save your final CSV in the same directory as your IPYNB


In [ ]:
#The following blocks of code will extract prenylation scores from a list and concatenate them into a string with a comma delimiter.

for i in final_FTase_csv.index:
    final_fscores = ''
    for element in final_FTase_csv.loc[i, 0]:
        final_fscores += f"{element},"
    final_fscores = final_fscores[:len(final_fscores)-1]
    final_FTase_csv.loc[i, 0] = final_fscores

for i in final_GGTase_csv.index:
    final_gscores = ''
    for element in final_GGTase_csv.loc[i, 0]:
        final_gscores += f"{element},"
    final_gscores = final_gscores[:len(final_gscores)-1]
    final_GGTase_csv.loc[i, 0] = final_gscores

In [ ]:
final_csv['Protein IDs'] = IDs
final_csv['FTase scores'] = final_FTase_csv.iloc[:, 0]
final_csv['GGTase scores'] = final_GGTase_csv.iloc[:, 0]
final_csv['sequences'] = final_seq_csv.iloc[:, 0]
final_csv = final_csv.reset_index(drop=True)
final_final_csv = pd.DataFrame()

In [ ]:
#The following blocks of code will drop any row that does not have atleast one prenylation score over -2.0.

for i in final_csv.index:
    
    FTase_scores = str(final_csv.loc[i, 'FTase scores']).split(',')
    GGTase_scores = str(final_csv.loc[i, 'GGTase scores']).split(',')
    
    for j in range(len(FTase_scores)):
        if len(FTase_scores) != len(GGTase_scores):
            pass
        
        elif float(FTase_scores[j]) > -2.0 or float(GGTase_scores[j]) > -2.0:
            final_final_csv = pd.concat([final_final_csv, final_csv.iloc[i]], ignore_index=True, axis = 1)
        else:
            pass

In [ ]:
#Cleans up outputfile

final_final_csv = final_final_csv.T
final_final_csv = final_final_csv.drop_duplicates(subset = ["Protein IDs"])

In [ ]:
#Save final output file

final_final_csv.to_csv(f'{filename}_{prenyldata_file}_CaaX_final.csv')

In [ ]:
#The following block of code will fetch N-terminomics data for all proteins in the output file

protein_ids = list(final_final_csv['Protein IDs'])

recordstermini = defaultdict(list)

for uniprot_id in protein_ids:
    url = f"https://topfind.clip.msl.ubc.ca/nterms/index?seq=&loc=&pos=&modifications=&species=3&ac={uniprot_id}&commit=Search"
    print(f"Fetching TopFIND data for {uniprot_id}...")
    
    response = requests.get(url, timeout=10)
    
    if response.status_code == 200:
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        
        
        
        soup_string = str(soup.get_text(separator='\n', strip=True))
        text = soup_string
        for m in re.finditer(r'Position: s*([0-9.-]+)', text):
            
            recordstermini[uniprot_id].append(f"{m.group()}")  
    else:
        print(f"HTTP Error {response.status_code} for {uniprot_id}")
        
    time.sleep(1)

In [ ]:
print(recordsx)

In [ ]:
#The following block of code will eliminate duplicate N-termini

for key, value in recordstermini.items():
    value = list(set(value))
    recordstermini.update({key:value})

#The following block of code cleans up output from TopFind

for key, value in recordstermini.items():
    for i in range(len(value)):
        if 'Position:' in value[i]:
            value[i] = int(value[i][10:])
        else:
            pass
        recordstermini.update({key:value})

In [ ]:
#These following blocks of code will truncate proteins sequences to show C-termini identified by TopFind.

epicdict = dict()

for key, value in recordstermini.items():
    for key1, value1, in recordsx.items():
        if key == key1.split('|')[1]:
            epicdict.update({key:str(value1.seq)})
        else:
            continue
            
anotherone = defaultdict(list)

for key, value in recordstermini.items():
    for key1, value1 in epicdict.items():
        if key == key1:
            for i in range(len(value)):
                anotherone[key].append(value1[:int(value[i])])
        else:
            pass

In [ ]:
#The following block of code will see if the C-termini identified by TopFind is a CaaX box.

chits = defaultdict(list)
for key, value in anotherone.items():
    for i in range(len(value)):
        for boop2 in Caa:
            if boop2 in value[i][-4:-1]:
                chits[key].append(value[i])
            else:
                pass

In [ ]:
print(chits)        #Print output

In [ ]:
df_topfind = pd.DataFrame.from_dict(pd.Series(recordstermini))
df_topfind.to_excel("terminiactually.xlsx")     #Save output file with C-termini identified by TopFind